# Deep Learning - Introduction à Pytorch


## TP2 : Fonctions Dérivables

Sylvain Lamprier (sylvain.lamprier@univ-angers.fr)

Supports adaptés de Nicolas Baskiotis (nicolas.baskiotis@sorbonne-univeriste.fr) et Benjamin Piwowarski (benjamin.piwowarski@sorbonne-universite.fr) -- MLIA/ISIR, Sorbonne Université

In [10]:
import torch
print("La version de torch est : ",torch.__version__)
print("Le calcul GPU est disponible ? ", torch.cuda.is_available())

import numpy as np
import sklearn
print("sklearn ",sklearn.__version__)
print("numpy ",np.__version__)

La version de torch est :  2.9.1+cu130
Le calcul GPU est disponible ?  True
sklearn  1.8.0
numpy  2.3.5


Au TP précédent, nous avons vu comment implémenter une regression linéaire en utilisant les structures Tensor de PyTorch. Cependant, nous exploitions pas du tout la puissance de PyTorch qui permet de faciliter le calcul des gradients via de l'auto-dérivation. Dans le TP précédent nous avions défini un algorithme spécifique à de la regression pour un modèle (linéaire) et un coût (moindres carrés) figés, en définissant à la main le gradient du coût global pour l'ensemble des paramètres. Ce mode de programmation est très peu modulaire et est très difficilement étendable à des architectures plus complexes. Sachant que l'objectif est de développer des architectures neuronales avec des nombreux modules neuronaux enchaînés, il n'est pas possible de travailler de cette façon.

Dans ce TP, nous allons voir comment décomposer les choses pour rendre le code plus facilement généralisable. L'objectif est de comprendre le fonctionnement interne de PyTorch (sans en utiliser encore les facilités offertes par l'utilisation d'un graphe de calcul), basé sur l'implémentation d'objets Function.  

## Fonctions


$\href{https://pytorch.org/docs/stable/}{\texttt{PyTorch}}$ utilise une classe abstraite $\href{https://pytorch.org/docs/stable/autograd.html#torch.autograd.Function}{\texttt{Function}}$ dont sont héritées toutes les fonctions et qui nécessite l'implémentation de ces deux méthodes :

- méthode $\texttt{forward(ctx, *inputs)}$ : calcule le résultat de l'application de la fonction
- méthode $\texttt{backward(ctx, *grad-outputs)}$ : calcule le gradient partiel par rapport à chaque entrée de la méthode $\texttt{forward}$; le nombre de $\texttt{grad-outputs}$ doit être égale aux nombre de sorties de $\texttt{forward}$ (pourquoi ?) et le nombre de  
sorties doit être égale aux nombres de $\texttt{inputs}$ de $\texttt{forward}$.


Pour des raisons d'implémentation, les deux méthodes doivent être statiques. Le premier paramètre $\texttt{ctx}$ permet de sauvegarder un contexte lors de la passe $\texttt{forward}$ (par exemple les tenseurs d'entrées) et il est passé lors de la passe $\texttt{backward}$ en paramètre afin de récupérer les valeurs. $\textbf{Attention : }$ le contexte doit être unique pour chaque appel de $\texttt{forward}$.

Compléter le code ci-dessous pour créer des modules MSE (coût moindres carrés) et Linéaire. Les deux cellules en dessous vous serviront à tester votre code: si tout se passe sans plantage, alors vos gradients semblent corrects. Utiliser bien les outils propres à pyTorch, en particulier des Tensor et pas des matrices numpy. Assurez vous que
vos fonctions prennent en entrée des batchs d’exemples (matrice 2D) et non un seul exemple (vecteur). N’hésiter pas à prendre un exemple et déterminer les dimensions des différentes matrices en jeu.  

In [11]:
import torch
from torch.autograd import Function
from torch.autograd import gradcheck


class Context:
    """Un objet contexte très simplifié pour simuler PyTorch

    Un contexte différent doit être utilisé à chaque forward
    """
    def __init__(self):
        self._saved_tensors = ()
    def save_for_backward(self, *args):
        self._saved_tensors = args
    @property
    def saved_tensors(self):
        return self._saved_tensors


class MSE(Function):
    """Début d'implementation de la fonction MSE"""
    @staticmethod
    def forward(ctx, yhat, y):
        ## Garde les valeurs nécessaires pour le backwards
        ctx.save_for_backward(yhat, y)

        # [[STUDENT]] Renvoyer la valeur de la fonction
        loss = ((yhat - y) ** 2).mean()
        return loss
        # [[/STUDENT]]

    @staticmethod
    def backward(ctx, grad_output):
        ## Calcul du gradient du module par rapport a chaque groupe d'entrées
        yhat, y = ctx.saved_tensors
       
        # [[STUDENT]] Renvoyer les deux dérivées partielles (par rapport à yhat et à y)
        N = yhat.numel()  # nombre d'éléments
        # Gradient par rapport à yhat et y
        grad_yhat = grad_output * 2 * (yhat - y) / N
        grad_y = grad_output * -2 * (yhat - y) / N
        return grad_yhat, grad_y
        # [[/STUDENT]]

# [[STUDENT]] Implémenter la fonction Linear(X, W, b)sur le même modèle que MSE
class Linear(Function):
    @staticmethod
    def forward(ctx, X, W, b):
        ctx.save_for_backward(X, W, b)
        return X @ W + b 

    @staticmethod
    def backward(ctx, grad_output):
        X, W, b = ctx.saved_tensors
        grad_X = grad_output @ W.T
        grad_W = X.T @ grad_output
        grad_b = grad_output.sum(0, keepdim=True)
        return grad_X, grad_W, grad_b

# [[/STUDENT]]

## Utile pour gradcheck
mse = MSE.apply
linear = Linear.apply






In [12]:
# Test du gradient de MSE
yhat = torch.randn(10,5, requires_grad=True, dtype=torch.float64)
y = torch.randn(10,5, requires_grad=True, dtype=torch.float64)
torch.autograd.gradcheck(mse, (yhat, y))

True

In [13]:
# Test du gradient de Linear (sur le même modèle que MSE)

x = torch.randn(13, 5,requires_grad=True,dtype=torch.float64)
w = torch.randn(5, 7,requires_grad=True,dtype=torch.float64)
b = torch.randn(7,requires_grad=True,dtype=torch.float64)
torch.autograd.gradcheck(linear,(x,w,b))

True

## Descente de Gradient

### Regression Linéaire

Compléter ci-dessous le code pour réaliser la même regression linéaire qu'au TP précédent, mais en utilisant les objets Function déclarés ci-dessus.

In [14]:
## Chargement des données California_Housing et transformation en tensor.
#figshare renvoyait 403 Forbidden, j'ai utilise open_ML

## Chargement des données California_Housing et transformation en tensor.
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler

housing = pd.read_csv("/home/erast/Files/M2/DeepLearning/housing.csv")  # fichier local

## Sélectionner uniquement les colonnes numériques (ignorer les colonnes non numériques)
numeric_cols = housing.select_dtypes(include='number').columns
X_df = housing[numeric_cols].drop(columns=['median_house_value'])
y_series = housing['median_house_value']

# Vérifier les valeurs manquantes et les remplir si besoin (ici on remplit par la moyenne)
X_df = X_df.fillna(X_df.mean())
y_series = y_series.fillna(y_series.mean())

# Normalisation des features
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X_df.values)

# Normalisation de la target
y_mean = y_series.mean()
y_std = y_series.std()
y_scaled = (y_series - y_mean) / y_std

# Conversion en tenseurs PyTorch
x = torch.tensor(X_scaled, dtype=torch.float, requires_grad=True)
y = torch.tensor(y_scaled.values, dtype=torch.float).view(-1, 1)

print("Nombre d'exemples : ", x.size(0), "Dimension : ", x.size(1))

#initialisation aléatoire de w et b
w = torch.randn(x.size(1),1, requires_grad=True)
b = torch.randn(1,1, requires_grad=True)


EPOCHS = 50000
EPS = 4e-3
for n_iter in range(EPOCHS):
    ## [[STUDENT]] Calcul du forward (loss), avec creation de nouveaux Context pour chaque module
    y_hat = linear(x,w,b)
    loss = mse(y_hat, y)

    
    # `loss` doit correspondre au coût MSE calculé à cette itération
    if n_iter % 100==0:
        print(f"Itérations {n_iter}: loss {loss}")

    ## [[STUDENT]] Calcul du backward (grad_w, grad_b)
    loss.backward()

    # [[/STUDENT]]

    ## [[STUDENT]] Mise à jour des paramètres du modèle
    with torch.no_grad():
        w -= EPS * w.grad
        b -= EPS * b.grad

        w.grad.zero_()
        b.grad.zero_()
        x.grad.zero_()
    # [[/STUDENT]]


Nombre d'exemples :  20640 Dimension :  8
Itérations 0: loss 11.805624008178711
Itérations 100: loss 2.008025884628296
Itérations 200: loss 1.037287712097168
Itérations 300: loss 0.8030303716659546
Itérations 400: loss 0.7154372334480286
Itérations 500: loss 0.6624091863632202
Itérations 600: loss 0.6213350296020508
Itérations 700: loss 0.5870067477226257
Itérations 800: loss 0.557726263999939
Itérations 900: loss 0.5325919985771179
Itérations 1000: loss 0.510952353477478
Itérations 1100: loss 0.49228137731552124
Itérations 1200: loss 0.4761420786380768
Itérations 1300: loss 0.4621672034263611
Itérations 1400: loss 0.4500473737716675
Itérations 1500: loss 0.43952077627182007
Itérations 1600: loss 0.4303649961948395
Itérations 1700: loss 0.4223913252353668
Itérations 1800: loss 0.41543859243392944
Itérations 1900: loss 0.40936893224716187
Itérations 2000: loss 0.4040645658969879
Itérations 2100: loss 0.3994242250919342
Itérations 2200: loss 0.3953607976436615
Itérations 2300: loss 0.391

### Regression Non Linéaire

Ajouter une classe Function Tanh sur le modèle des classe déclarées ci-dessus et appliquer une descente de gradient sur le problème précédent qui utilise un réseau de neurones à une couche cachée de 10 neurones. 

In [15]:
# [[STUDENT]] Implémenter la fonction Tanh(X)sur le même modèle que MSE et Linear
class Tanh(Function):
    @staticmethod
    def forward(ctx, x):
        exp_x = torch.exp(x)
        exp_neg_x = torch.exp(-x)
        y = (exp_x - exp_neg_x) / (exp_x + exp_neg_x)

        ctx.save_for_backward(y)
        return y

    @staticmethod
    def backward(ctx, grad_output):
        (y,) = ctx.saved_tensors

        grad_x = grad_output * (1 - y ** 2)
        return grad_x
    
# [[/STUDENT]]

## Utile pour gradcheck
tanh=Tanh.apply

In [16]:
# Test du gradient de Tanh (sur le même modèle que MSE)

x = torch.randn(13, 5,requires_grad=True,dtype=torch.float64)
torch.autograd.gradcheck(tanh,x)

True

In [18]:
## Chargement des données California_Housing et transformation en tensor.

housing = pd.read_csv("/home/erast/Files/M2/DeepLearning/housing.csv")  # fichier local

## Sélectionner uniquement les colonnes numériques (ignorer les colonnes non numériques)
numeric_cols = housing.select_dtypes(include='number').columns
X_df = housing[numeric_cols].drop(columns=['median_house_value'])
y_series = housing['median_house_value']

# Vérifier les valeurs manquantes et les remplir si besoin (ici on remplit par la moyenne)
X_df = X_df.fillna(X_df.mean())
y_series = y_series.fillna(y_series.mean())

# Normalisation des features
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X_df.values)

# Normalisation de la target
y_mean = y_series.mean()
y_std = y_series.std()
y_scaled = (y_series - y_mean) / y_std

# Conversion en tenseurs PyTorch
x = torch.tensor(X_scaled, dtype=torch.float, requires_grad=True)
y = torch.tensor(y_scaled.values, dtype=torch.float).view(-1, 1)
print("Nombre d'exemples : ",x.size(0), "Dimension : ",x.size(1))

# [[STUDENT]] Implémenter la descente de gradient précédente selon un réseau à une couche cachée de 10 neurones

HIDDEN_SIZE = 10

# Initialisation des paramètres
w1 = torch.randn(x.size(1), HIDDEN_SIZE, requires_grad=True)
b1 = torch.randn(1, HIDDEN_SIZE, requires_grad=True)

w2 = torch.randn(HIDDEN_SIZE, 1, requires_grad=True)
b2 = torch.randn(1, 1, requires_grad=True)


EPOCHS = 50000
EPS = 1e-3

for n_iter in range(EPOCHS):

    # -------- Forward --------
    z1 = linear(x, w1, b1)      # couche cachée (pré-activation)
    h1 = tanh(z1)               # activation tanh
    y_hat = linear(h1, w2, b2)  # sortie
    loss = mse(y_hat, y)        # loss MSE

    if n_iter % 100 == 0:
        print(f"Itération {n_iter}: loss {loss.item()}")

    # -------- Backward --------
    loss.backward()

    # -------- Mise à jour des paramètres --------
    with torch.no_grad():
        w1 -= EPS * w1.grad
        b1 -= EPS * b1.grad
        w2 -= EPS * w2.grad
        b2 -= EPS * b2.grad

        # remise à zéro des gradients
        w1.grad.zero_()
        b1.grad.zero_()
        w2.grad.zero_()
        b2.grad.zero_()
        x.grad.zero_()
    
# [[/STUDENT]]

Nombre d'exemples :  20640 Dimension :  8
Itération 0: loss 10.891458511352539
Itération 100: loss 3.870492935180664
Itération 200: loss 2.4153218269348145
Itération 300: loss 1.9213097095489502
Itération 400: loss 1.6681981086730957
Itération 500: loss 1.5070992708206177
Itération 600: loss 1.3920621871948242
Itération 700: loss 1.3034863471984863
Itération 800: loss 1.231524109840393
Itération 900: loss 1.1708128452301025
Itération 1000: loss 1.1182506084442139
Itération 1100: loss 1.0719342231750488
Itération 1200: loss 1.0306214094161987
Itération 1300: loss 0.9934497475624084
Itération 1400: loss 0.9597871899604797
Itération 1500: loss 0.9291483759880066
Itération 1600: loss 0.9011460542678833
Itération 1700: loss 0.8754640817642212
Itération 1800: loss 0.8518378734588623
Itération 1900: loss 0.8300431370735168
Itération 2000: loss 0.8098875284194946
Itération 2100: loss 0.7912048101425171
Itération 2200: loss 0.7738498449325562
Itération 2300: loss 0.7576960325241089
Itération 24